# Component 3: Sinusoidal Timestep Embedding

In the diffusion process, the model must denoise action trajectories over many timesteps (e.g., 100 steps from pure noise to a clean trajectory). At each step $t$, the model needs to know **how noisy** the current input is, because the denoising strategy at $t=95$ (very noisy) is completely different from $t=5$ (almost clean).

We encode the scalar timestep $t$ into a high-dimensional vector using **sinusoidal positional embeddings** -- the same trick used in the original Transformer paper. The embedding uses sine and cosine functions at different frequencies:

$$\text{PE}(t, 2i) = \sin\left(\frac{t}{10000^{2i/d}}\right)$$
$$\text{PE}(t, 2i+1) = \cos\left(\frac{t}{10000^{2i/d}}\right)$$

where $d$ is the embedding dimension and $i$ is the dimension index.

**Why sinusoidal?** Three key properties:
1. **Unique**: Every timestep gets a distinct embedding vector
2. **Smooth**: Nearby timesteps have similar embeddings (high cosine similarity)
3. **Multi-scale**: Low-frequency channels capture coarse time progression, high-frequency channels capture fine-grained differences

In [ ]:
!pip install -q torch torchvision matplotlib numpy 2>&1 | tail -3

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
import torch.nn as nn

# ============================================================
# Build the diffusion timestep encoder from scratch
# (same architecture as used in the real diffusion policy)
# ============================================================

class SinusoidalPosEmb(nn.Module):
    """Sinusoidal positional embedding for diffusion timesteps."""
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, x):
        half_dim = self.dim // 2
        emb = torch.log(torch.tensor(10000.0)) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=x.device, dtype=torch.float32) * -emb)
        emb = x.unsqueeze(-1).float() * emb.unsqueeze(0)
        emb = torch.cat([emb.sin(), emb.cos()], dim=-1)
        return emb


class DiffusionStepEncoder(nn.Module):
    """Full timestep encoder: sinusoidal embedding -> MLP.
    
    This matches the architecture used in the real diffusion policy UNet.
    The sinusoidal embedding provides a fixed encoding, then a 2-layer MLP
    learns a nonlinear transformation of the timestep.
    """
    def __init__(self, dim=128):
        super().__init__()
        self.net = nn.Sequential(
            SinusoidalPosEmb(dim),
            nn.Linear(dim, dim * 4),
            nn.Mish(),
            nn.Linear(dim * 4, dim),
        )

    def forward(self, x):
        return self.net(x)


# Create the encoder (same as what lives inside the UNet)
embed_dim_policy = 128
step_encoder = DiffusionStepEncoder(dim=embed_dim_policy).to(device)

print("Diffusion step encoder built from scratch!")
print(f"Architecture: SinusoidalPosEmb({embed_dim_policy}) -> Linear -> Mish -> Linear")
print(f"Total parameters: {sum(p.numel() for p in step_encoder.parameters()):,}")
print(f"\nThis is the same architecture used inside the diffusion policy UNet.")

## Building Sinusoidal Embeddings from Scratch

Let us implement the sinusoidal position embedding formula step by step, then visualize what these embeddings look like.

In [ ]:
def sinusoidal_embedding(timesteps, dim):
    """
    Compute sinusoidal positional embeddings.
    
    Args:
        timesteps: 1D tensor of timestep values, shape (T,)
        dim: embedding dimension (must be even)
    
    Returns:
        embeddings: shape (T, dim)
    """
    assert dim % 2 == 0, "Embedding dimension must be even"
    half_dim = dim // 2
    
    # Compute the frequency for each dimension
    # freq_i = 1 / 10000^(2i/d)
    i = torch.arange(half_dim, dtype=torch.float32)
    frequencies = 1.0 / (10000 ** (2 * i / dim))  # shape: (half_dim,)
    
    print(f"  Frequencies range: [{frequencies[0]:.6f}, ..., {frequencies[-1]:.10f}]")
    print(f"  Highest frequency period: {2 * np.pi / frequencies[0]:.1f} timesteps")
    print(f"  Lowest frequency period:  {2 * np.pi / frequencies[-1]:.1f} timesteps")
    
    # Outer product: timesteps x frequencies -> (T, half_dim)
    angles = timesteps.unsqueeze(1).float() * frequencies.unsqueeze(0)  # (T, half_dim)
    
    # Apply sin to even indices, cos to odd indices
    sin_emb = torch.sin(angles)  # (T, half_dim)
    cos_emb = torch.cos(angles)  # (T, half_dim)
    
    # Interleave: [sin_0, cos_0, sin_1, cos_1, ...]
    embeddings = torch.zeros(len(timesteps), dim)
    embeddings[:, 0::2] = sin_emb
    embeddings[:, 1::2] = cos_emb
    
    return embeddings


# Compute embeddings for t=0..99
num_timesteps = 100
embed_dim = 128
timesteps = torch.arange(num_timesteps)

print(f"Computing sinusoidal embeddings for {num_timesteps} timesteps, dim={embed_dim}")
embeddings = sinusoidal_embedding(timesteps, embed_dim)
print(f"\nEmbedding matrix shape: {embeddings.shape}")
print(f"Each timestep is represented by a {embed_dim}-dimensional vector")
print(f"\nEmbedding for t=0:  [{embeddings[0, :6].numpy().round(3)} ...]")
print(f"Embedding for t=50: [{embeddings[50, :6].numpy().round(3)} ...]")
print(f"Embedding for t=99: [{embeddings[99, :6].numpy().round(3)} ...]")

In [ ]:
# Visualization 1: Embedding matrix as a heatmap
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Full heatmap
im = axes[0].imshow(embeddings.numpy(), aspect="auto", cmap="RdBu_r",
                     vmin=-1, vmax=1, interpolation="nearest")
axes[0].set_xlabel("Embedding Dimension", fontsize=12)
axes[0].set_ylabel("Timestep t", fontsize=12)
axes[0].set_title("Sinusoidal Timestep Embeddings\n(100 timesteps x 128 dimensions)",
                   fontsize=13, fontweight="bold")
plt.colorbar(im, ax=axes[0], label="Value")

# Annotate
axes[0].annotate("Low freq\n(slow change)", xy=(embed_dim-1, 50),
                 xytext=(embed_dim+15, 50), fontsize=9,
                 arrowprops=dict(arrowstyle="->", color="black"),
                 ha="left", va="center")
axes[0].annotate("High freq\n(fast change)", xy=(0, 50),
                 xytext=(-35, 50), fontsize=9,
                 arrowprops=dict(arrowstyle="->", color="black"),
                 ha="right", va="center")

# Cosine similarity matrix
emb_norm = embeddings / embeddings.norm(dim=1, keepdim=True)
cos_sim = emb_norm @ emb_norm.T  # (100, 100)

im2 = axes[1].imshow(cos_sim.numpy(), cmap="viridis", aspect="equal")
axes[1].set_xlabel("Timestep t₁", fontsize=12)
axes[1].set_ylabel("Timestep t₂", fontsize=12)
axes[1].set_title("Cosine Similarity Between\nTimestep Embeddings",
                   fontsize=13, fontweight="bold")
plt.colorbar(im2, ax=axes[1], label="Cosine Similarity")

plt.tight_layout()
plt.show()

# Print similarity for specific pairs
print("Cosine similarity between nearby timesteps:")
print(f"  sim(t=0, t=1)  = {cos_sim[0, 1].item():.4f}  (adjacent)")
print(f"  sim(t=0, t=5)  = {cos_sim[0, 5].item():.4f}  (close)")
print(f"  sim(t=0, t=50) = {cos_sim[0, 50].item():.4f}  (far)")
print(f"  sim(t=0, t=99) = {cos_sim[0, 99].item():.4f}  (opposite ends)")

In [ ]:
# Visualization 2: Individual frequency channels across timesteps
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle("Individual Embedding Channels Across Timesteps",
             fontsize=14, fontweight="bold")

channel_indices = [0, 10, 40, 126]  # From high to low frequency
colors = ["#D97757", "#5B8CB8", "#7DA488", "#9B7EC8"]
t_range = np.arange(num_timesteps)

for idx, (ch, color) in enumerate(zip(channel_indices, colors)):
    ax = axes[idx // 2, idx % 2]
    values = embeddings[:, ch].numpy()
    ax.plot(t_range, values, color=color, linewidth=2)
    ax.fill_between(t_range, values, alpha=0.15, color=color)
    ax.set_xlabel("Timestep t")
    ax.set_ylabel("Value")
    kind = "sin" if ch % 2 == 0 else "cos"
    freq_idx = ch // 2
    freq_val = 1.0 / (10000 ** (2 * freq_idx / embed_dim))
    period = 2 * np.pi / freq_val
    ax.set_title(f"Channel {ch} ({kind}, freq_idx={freq_idx})\n"
                 f"Period = {period:.1f} timesteps", fontsize=11)
    ax.set_ylim(-1.2, 1.2)
    ax.axhline(y=0, color="gray", linestyle="--", alpha=0.3)
    ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

print("Key observation: Low-index channels oscillate rapidly (high frequency),")
print("allowing the model to distinguish adjacent timesteps.")
print("High-index channels oscillate slowly (low frequency),")
print("capturing the overall progression from noisy (t~100) to clean (t~0).")

In [ ]:
# Compare with the MLP-enhanced timestep encoder we built above
print("=" * 60)
print("Comparing Raw Sinusoidal vs MLP-Enhanced Embeddings")
print("=" * 60)

print(f"\nDiffusion step encoder architecture:")
print(step_encoder)

# Pass a few timesteps through the MLP-enhanced encoder
test_timesteps = torch.tensor([0, 25, 50, 75, 99], device=device)
with torch.no_grad():
    policy_embeddings = step_encoder(test_timesteps)

print(f"\nMLP-enhanced output shape: {policy_embeddings.shape}")
print(f"Output dim: sinusoidal {embed_dim_policy}d -> MLP -> {policy_embeddings.shape[-1]}d")

print("\nThe MLP on top of the sinusoidal embedding allows the network")
print("to learn a nonlinear transformation of the timestep,")
print("which can be more expressive than raw sinusoids.")

# Cosine similarity between MLP-enhanced embeddings
pe_norm = policy_embeddings / policy_embeddings.norm(dim=1, keepdim=True)
pe_sim = pe_norm @ pe_norm.T
print("\nCosine similarity (MLP-enhanced encoder, random init):")
for i, ti in enumerate(test_timesteps):
    for j, tj in enumerate(test_timesteps):
        if j > i:
            print(f"  sim(t={ti.item()}, t={tj.item()}) = {pe_sim[i,j].item():.4f}")

print("\nNote: Since this encoder has random MLP weights (not trained),")
print("the similarity structure may differ from a trained model.")
print("After training, the MLP learns to produce embeddings that")
print("help the UNet distinguish noise levels optimally.")

## TODO Exercise

**What if you used a random embedding instead of sinusoidal? Plot the similarity matrix and compare.**

```python
# TODO: Create random embeddings for 100 timesteps
# random_embeddings = torch.randn(100, 128)
# 
# 1. Plot the cosine similarity matrix for random embeddings
# 2. Compare with the sinusoidal similarity matrix side-by-side
# 3. What property does the sinusoidal embedding have that random lacks?
#    (Hint: look at the diagonal bands in the similarity matrix)
# 
# 4. Bonus: try a LEARNED embedding (nn.Embedding(100, 128)) initialized
#    randomly. What would happen during training? Why might sinusoidal
#    be a better initialization than random?
```